In [20]:
%load_ext autoreload
%autoreload 2                    

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
import gym
import numpy as np
import torch

from agents.pg_agent import PGAgent
from infrastructure import utils
from infrastructure import pytorch_util as ptu

In [22]:
env_name = 'CartPole-v0'
env = gym.make(env_name, render_mode=None)
discrete = isinstance(env.action_space, gym.spaces.Discrete)

max_ep_len = env.spec.max_episode_steps
ob_dim = env.observation_space.shape[0]
ac_dim = env.action_space.n if discrete else env.action_space.shape[0]

print(f'ob_dim={ob_dim}, ac_dim={ac_dim}, discrete={discrete}, max_ep_len={max_ep_len}')

ob_dim=4, ac_dim=2, discrete=True, max_ep_len=200


/Users/jasonkrone/Developer/homework_spring2026/hw2/.venv/lib/python3.10/site-packages/gym/envs/registration.py:593: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(
/Users/jasonkrone/Developer/homework_spring2026/hw2/.venv/lib/python3.10/site-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/Users/jasonkrone/Developer/homework_spring2026/hw2/.venv/lib/python3.10/site-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


In [23]:
ptu.init_gpu(use_gpu=False)

agent = PGAgent(
    ob_dim,
    ac_dim,
    discrete,
    n_layers=2,
    layer_size=64,
    gamma=1.0,
    learning_rate=5e-3,
    use_baseline=False,
    use_reward_to_go=False,
    normalize_advantages=False,
    baseline_learning_rate=None,
    baseline_gradient_steps=None,
    gae_lambda=None,
)

Using CPU.


In [24]:
agent.actor.get_action = lambda x: env.action_space.sample()

In [25]:
batch_size = 1000
trajs, envsteps_this_batch = utils.sample_trajectories(
    env, agent.actor, batch_size, max_ep_len
)
print(f'collected {len(trajs)} trajectories, {envsteps_this_batch} total steps')

collected 46 trajectories, 1010 total steps


/Users/jasonkrone/Developer/homework_spring2026/hw2/.venv/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:241: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


In [26]:
trajs_dict = {k: [traj[k] for traj in trajs] for k in trajs[0]}
print({k: len(v) for k, v in trajs_dict.items()})

{'observation': 46, 'image_obs': 46, 'reward': 46, 'action': 46, 'next_observation': 46, 'terminal': 46}


In [27]:
env.action_space

Discrete(2)

In [28]:
env.state

(-0.06892750934076418,
 -0.20203993774834278,
 0.22207025800899605,
 0.8387390627577201)

In [29]:
len(trajs_dict["observation"]), len(trajs_dict["observation"][0])

(46, 13)

In [30]:
trajs_dict["observation"][0]

array([[ 2.3657048e-02,  5.4090394e-04, -2.8794069e-02, -1.6282642e-02],
       [ 2.3667866e-02,  1.9606371e-01, -2.9119721e-02, -3.1790957e-01],
       [ 2.7589140e-02,  3.9158803e-01, -3.5477914e-02, -6.1963171e-01],
       [ 3.5420902e-02,  1.9697911e-01, -4.7870547e-02, -3.3833033e-01],
       [ 3.9360482e-02,  3.9274839e-01, -5.4637153e-02, -6.4571643e-01],
       [ 4.7215451e-02,  1.9842865e-01, -6.7551479e-02, -3.7072709e-01],
       [ 5.1184025e-02,  4.3282006e-03, -7.4966021e-02, -1.0008549e-01],
       [ 5.1270589e-02,  2.0044002e-01, -7.6967731e-02, -4.1544640e-01],
       [ 5.5279389e-02,  3.9656365e-01, -8.5276663e-02, -7.3136705e-01],
       [ 6.3210659e-02,  5.9275407e-01, -9.9904001e-02, -1.0496243e+00],
       [ 7.5065747e-02,  7.8904921e-01, -1.2089649e-01, -1.3719218e+00],
       [ 9.0846725e-02,  9.8545766e-01, -1.4833492e-01, -1.6998410e+00],
       [ 1.1055588e-01,  1.1819452e+00, -1.8233174e-01, -2.0347865e+00]],
      dtype=float32)

In [31]:
trajs_dict.keys()

dict_keys(['observation', 'image_obs', 'reward', 'action', 'next_observation', 'terminal'])

In [32]:
trajs_dict["reward"][0]

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32)

In [33]:
trajs_dict["observation"][2].shape

(12, 4)

In [34]:
np.concatenate(trajs_dict["observation"], axis=0).shape

(1010, 4)

In [35]:
q_values = agent._calculate_q_vals(trajs_dict["reward"])

In [36]:
q_values[0]

array([13., 13., 13., 13., 13., 13., 13., 13., 13., 13., 13., 13., 13.])

In [37]:
np.arange(10)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [38]:
agent.gamma

1.0

In [41]:
np.zeros_like(np.arange(3))

array([0, 0, 0])

In [44]:
np.std(np.concatenate(q_values) )

12.135584878815935